In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import gc, os, zipfile, getpass
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import cohen_kappa_score
from scipy.special import expit
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from arabert.preprocess import ArabertPreprocessor

# ==========================================
# 1. LOAD DỮ LIỆU SẠCH (Từ file 02)
# ==========================================
PROCESSED_DIR = '../data/processed/'
MODEL_NAME    = "UBC-NLP/ARBERT"
NUM_CLASSES   = 19

# Load dữ liệu đã làm sạch[cite: 7]
df_all = pd.read_parquet(PROCESSED_DIR + 'all_cleaned.parquet')
print(f"✅ Load xong: {len(df_all)} mẫu | Labels: {sorted(df_all['label'].unique())}")

# Tải Blind Test[cite: 7]
access_token = getpass.getpass("🔒 Nhập Hugging Face Token: ")
barec_sent   = load_dataset("CAMeL-Lab/BAREC-Shared-Task-2026-BlindTest-sent", token=access_token)
df_blind_test = barec_sent['train'].to_pandas()

arabert_prep  = ArabertPreprocessor(model_name=MODEL_NAME)
df_blind_test['Sentence_Normalized'] = df_blind_test['Sentence'].apply(
    lambda x: arabert_prep.preprocess(str(x))
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["Sentence_Normalized"],
        padding="max_length", truncation=True, max_length=128
    )

hf_blind_test      = Dataset.from_pandas(df_blind_test[['Sentence_Normalized']])
tokenized_blind_test = hf_blind_test.map(tokenize_function, batched=True)
tokenized_blind_test.set_format(type='torch', columns=['input_ids', 'attention_mask'])

# ==========================================
# 2. ĐỊNH NGHĨA CORAL TRAINER & METRICS 
# ==========================================
class CORALTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Fix an toàn để lấy nhãn[cite: 6]
        labels = inputs.pop("labels", inputs.get("label"))
        outputs = model(**inputs)
        logits = outputs.logits
        device = logits.device
        
        # Ma trận Target cho 18 lằn ranh[cite: 6]
        target = torch.zeros_like(logits, device=device)
        for k in range(logits.shape[1]):
            target[:, k] = (labels > k).float()
            
        loss_per_sample = F.binary_cross_entropy_with_logits(
            logits, target, reduction='none'
        ).mean(dim=1)
        
        # Ép trọng số mẫu[cite: 6]
        if self.class_weights is not None:
            w = self.class_weights.to(device)[labels]
            loss = (loss_per_sample * w).mean()
        else:
            loss = loss_per_sample.mean()
            
        return (loss, outputs) if return_outputs else loss

def compute_metrics_coral(eval_pred):
    logits, labels = eval_pred
    preds_binary = logits > 0
    pred_labels = preds_binary.sum(axis=1)
    qwk = cohen_kappa_score(labels, pred_labels, weights='quadratic')
    return {"qwk": qwk}

# ==========================================
# 3. K-FOLD TRAINING
# ==========================================
n_splits = 5
skf      = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_logits        = np.zeros((len(df_all), NUM_CLASSES - 1))
oof_true_labels   = np.zeros(len(df_all), dtype=np.int64)
test_logits_folds = []

# Chỉ lấy cột label gốc cho CORALTrainer chuẩn
COLS_TRAIN = ['Sentence_Normalized', 'label']
FMT_COLS   = ['input_ids', 'attention_mask', 'label']

print("\n========================================================")
print("🌟 BẮT ĐẦU CHIẾN DỊCH HUẤN LUYỆN (ARBERT + CORAL TRAINER) 🌟")
print("========================================================")

for fold, (train_idx, val_idx) in enumerate(skf.split(df_all, df_all['label'])):
    print(f"\n🚀 FOLD {fold+1}/{n_splits}")

    train_fold = df_all.iloc[train_idx]
    val_fold   = df_all.iloc[val_idx]

    cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_fold['label'].values)
    cw = np.clip(cw, 0.5, 5.0)
    cw = cw / cw.mean()
    class_weights_tensor = torch.tensor(cw, dtype=torch.float32)

    hf_train = Dataset.from_pandas(train_fold[COLS_TRAIN])
    hf_val   = Dataset.from_pandas(val_fold[COLS_TRAIN])

    tokenized_train = hf_train.map(tokenize_function, batched=True)
    tokenized_val   = hf_val.map(tokenize_function, batched=True)

    tokenized_train.set_format(type='torch', columns=FMT_COLS)
    tokenized_val.set_format(type='torch', columns=FMT_COLS)

    # Sử dụng AutoModelForSequenceClassification chuẩn[cite: 6]
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=18)
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS, 
        r=16, 
        lora_alpha=32, 
        lora_dropout=0.05,
        target_modules=["query", "value", "dense"] 
    )
    model = get_peft_model(model, lora_config)

    training_args = TrainingArguments(
        output_dir=f"../saved_models/arbert_coral_fold_{fold+1}_seed42",
        num_train_epochs=5,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=16,
        learning_rate=3e-4,
        weight_decay=0.01,
        bf16=True, fp16=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="qwk",
        greater_is_better=True,
        logging_strategy="epoch",
        seed=42,
        save_safetensors=False, 
    )

    trainer = CORALTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics_coral,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        class_weights=class_weights_tensor,
    )

    trainer.train()

    # OOF inference
    # Remove label để tắt tính toán loss dư thừa của Trainer
    tokenized_val_eval = tokenized_val.remove_columns(["label"])
    val_pred = trainer.predict(tokenized_val_eval)
    oof_logits[val_idx]      = val_pred.predictions[:, :18]
    oof_true_labels[val_idx] = val_fold['label'].values 

    # Blind test inference
    test_pred = trainer.predict(tokenized_blind_test)
    test_logits_folds.append(test_pred.predictions[:, :18])

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

avg_test_logits = np.mean(test_logits_folds, axis=0)

# ==========================================
# 4. OOF REPORT + DISTRIBUTION ALIGNMENT
# ==========================================
oof_preds = (oof_logits > 0).sum(axis=1)
oof_qwk   = cohen_kappa_score(oof_true_labels, oof_preds, weights='quadratic')
print(f"\n📊 OOF QWK (raw, trước alignment): {oof_qwk:.4f}")

test_probs        = expit(avg_test_logits)
continuous_scores = test_probs.sum(axis=1)

label_dist    = pd.Series(oof_true_labels).value_counts(normalize=True).sort_index()
n_test        = len(avg_test_logits)
target_counts = (label_dist * n_test).round().astype(int)
target_counts.iloc[-1] += n_test - target_counts.sum()

sorted_idx  = np.argsort(continuous_scores)
final_preds = np.zeros(n_test, dtype=int)
curr        = 0

for label in range(NUM_CLASSES):
    count = int(target_counts.get(label, 0))
    if count > 0:
        final_preds[sorted_idx[curr:curr+count]] = label + 1
        curr += count

submission_df = pd.DataFrame({
    'Sentence ID': df_blind_test['ID'].values,
    'Prediction':  final_preds
})
submission_df.to_csv('prediction', index=False, lineterminator='\n')

with zipfile.ZipFile('prediction_arbert_coral.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('prediction', arcname='prediction')

print("📦 ĐÃ XUẤT FILE: prediction_arbert_coral.zip")

✅ Load xong: 65894 mẫu | Labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]


Map:   0%|          | 0/8077 [00:00<?, ? examples/s]


🌟 BẮT ĐẦU CHIẾN DỊCH HUẤN LUYỆN (ARBERT + CORAL TRAINER) 🌟

🚀 FOLD 1/5


Map:   0%|          | 0/52715 [00:00<?, ? examples/s]

Map:   0%|          | 0/13179 [00:00<?, ? examples/s]

...

  0%|          | 0/824 [00:00<?, ?it/s]

{'eval_loss': 0.06193572282791138, 'eval_qwk': 0.8165847144248011, 'eval_runtime': 55.0147, 'eval_samples_per_second': 239.536, 'eval_steps_per_second': 14.978, 'epoch': 5.0}
{'train_runtime': 3568.481, 'train_samples_per_second': 73.863, 'train_steps_per_second': 2.308, 'train_loss': 0.05084188585217678, 'epoch': 5.0}


  0%|          | 0/824 [00:00<?, ?it/s]

  0%|          | 0/505 [00:00<?, ?it/s]


📊 OOF QWK (raw, trước alignment): 0.8216
📦 ĐÃ XUẤT FILE: prediction_arbert_coral.zip
